Advanced Ensemble Comparison
-Dataset: features_v2
-Same split
-RF vs XGBoost
-Purpose: compare ensemble techniques

To ensure fair algorithmic comparison, Random Forest and XGBoost were trained using the same feature set (features_v2) and identical time-aware train/test splits. Baseline models were evaluated separately using an earlier feature set (features_v1) to quantify the impact of feature engineering.

In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb

import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/processed/features_v2.csv")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

In [3]:
split_date = df["date"].quantile(0.8)

train = df[df["date"] < split_date].copy()
test  = df[df["date"] >= split_date].copy()

print("Train size:", train.shape)
print("Test size:", test.shape)

Train size: (8723, 20)
Test size: (2185, 20)


In [4]:
target_finish = "positionOrder"
target_lap = "avgLapTime_s"
target_points = "points"

In [5]:
drop_cols = [
    "raceId",
    "driverId",
    "constructorId",
    "date"
]

features = [
    col for col in df.columns
    if col not in drop_cols + [target_finish, target_lap, target_points]
]

print("Number of features:", len(features))

Number of features: 13


In [6]:
def prepare_data(target):
    X_train = train[features]
    X_test = test[features]
    
    y_train = train[target]
    y_test = test[target]
    
    return X_train, X_test, y_train, y_test

In [7]:
def evaluate_models(target_name, target_column):
    
    X_train, X_test, y_train, y_test = prepare_data(target_column)
    
    results = []
    
    # -----------------------------
    # Random Forest
    # -----------------------------
    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    )
    
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)
    
    rf_mae = mean_absolute_error(y_test, rf_preds)
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
    rf_r2 = r2_score(y_test, rf_preds)
    
    results.append(["RandomForest_v2", target_name, rf_mae, rf_rmse, rf_r2])
    
    
    # -----------------------------
    # XGBoost
    # -----------------------------
    xgb_model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror"
    )
    
    xgb_model.fit(X_train, y_train)
    xgb_preds = xgb_model.predict(X_test)
    
    xgb_mae = mean_absolute_error(y_test, xgb_preds)
    xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
    xgb_r2 = r2_score(y_test, xgb_preds)
    
    results.append(["XGBoost_v2", target_name, xgb_mae, xgb_rmse, xgb_r2])
    
    return results

In [8]:
all_results = []

all_results += evaluate_models("FinishPosition", target_finish)
all_results += evaluate_models("AvgLapTime", target_lap)
all_results += evaluate_models("ConstructorPoints", target_points)


In [9]:
results_df = pd.DataFrame(
    all_results,
    columns=["model", "target", "MAE", "RMSE", "R2"]
)

print(results_df)


             model             target        MAE       RMSE        R2
0  RandomForest_v2     FinishPosition   3.197246   4.106062  0.474129
1       XGBoost_v2     FinishPosition   3.198001   4.228097  0.442406
2  RandomForest_v2         AvgLapTime  12.220768  26.894941  0.155767
3       XGBoost_v2         AvgLapTime  12.186096  26.964126  0.151418
4  RandomForest_v2  ConstructorPoints   3.171436   4.681063  0.587355
5       XGBoost_v2  ConstructorPoints   3.123838   4.724659  0.579633


To ensure fair algorithmic comparison, Random Forest and XGBoost models were trained on the same engineered feature set (features_v2) using an identical time-aware 80/20 train–test split. Performance was evaluated using MAE, RMSE and R² across three prediction targets.

In [10]:
os.makedirs("../reports", exist_ok=True)

results_df.to_csv("../reports/stage3_model_comparison.csv", index=False)

print("Model comparison saved to /reports/")


Model comparison saved to /reports/
